In [5]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ARCHIVE_PATH = Path("/content/drive/MyDrive/Avenue_Dataset.zip")
EXTRACT_ROOT = Path("/content/avenue_dataset")
!unzip -q "{ARCHIVE_PATH}" -d "{EXTRACT_ROOT}"

In [14]:
import json
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import Video

# Change these
clip_source = Path("/content/avenue_dataset/Avenue Dataset/testing_videos/01.avi")
json_path = Path("/content/drive/MyDrive/STG-NF/Avenue_dataset/pose/test/01_0001_alphapose-results.json")
out_path = Path("/content/01_0028_pose_video.mp4")

# COCO-17 skeleton
SKELETON = [
    (0,1), (0,2), (1,3), (2,4),
    (5,6), (5,7), (7,9), (6,8), (8,10),
    (5,11), (6,12), (11,12),
    (11,13), (13,15), (12,14), (14,16)
]

def natural_key(p):
    stem = p.stem
    return int(stem) if stem.isdigit() else stem

def color_from_id(idx):
    idx = int(idx)
    return (
        (37 * idx) % 255,
        (97 * idx) % 255,
        (173 * idx) % 255,
    )

def frame_name_candidates(frame_index, clip_prefix):
    numbers = [
        str(frame_index),
        str(frame_index + 1),
        f"{frame_index:04d}",
        f"{frame_index + 1:04d}",
        f"{frame_index:05d}",
        f"{frame_index + 1:05d}",
        f"{frame_index:06d}",
        f"{frame_index + 1:06d}",
    ]
    names = set()
    for number in numbers:
        names.add(f"{number}.jpg")
        names.add(f"{number}.jpeg")
        names.add(f"{number}.png")
        names.add(f"{number}.avi")
        if clip_prefix:
            names.add(f"{clip_prefix}_{number}.jpg")
            names.add(f"{clip_prefix}_{number}.jpeg")
            names.add(f"{clip_prefix}_{number}.png")
    return list(names)

# Load AlphaPose rows
rows = json.loads(json_path.read_text())

# Group detections by frame image
by_image = defaultdict(list)
for row in rows:
    by_image[Path(row["image_id"]).name].append(row)

if clip_source.is_dir():
    # Read ordered frame list from extracted images
    frame_files = sorted(
        [p for p in clip_source.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]],
        key=natural_key
    )
    first = cv2.imread(str(frame_files[0]))
    h, w = first.shape[:2]
    frame_iter = ((frame_path.name, cv2.imread(str(frame_path))) for frame_path in frame_files)
else:
    # Read frames directly from a video file such as .avi
    video_capture = cv2.VideoCapture(str(clip_source))
    if not video_capture.isOpened():
        raise ValueError(f"Could not open video: {clip_source}")
    width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    w, h = width, height
    clip_prefix = clip_source.stem

    def video_frames():
        frame_index = 0
        while True:
            success, frame = video_capture.read()
            if not success:
                break
            yield frame_name_candidates(frame_index, clip_prefix), frame
            frame_index += 1
        video_capture.release()

    frame_iter = video_frames()

# Init video writer
writer = cv2.VideoWriter(
    str(out_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    25,
    (w, h)
)

KP_THR = 0.25

for frame_names, img in frame_iter:
    if img is None:
        continue

    if isinstance(frame_names, str):
        candidate_names = [frame_names]
    else:
        candidate_names = frame_names

    detections = []
    for candidate_name in candidate_names:
        detections = by_image.get(candidate_name, [])
        if detections:
            break

    for det in detections:
        kp = np.array(det["keypoints"], dtype=np.float32).reshape(-1, 3)
        bbox = det.get("box", None)
        track_id = int(det.get("idx", -1))
        color = color_from_id(track_id if track_id >= 0 else 0)

        # draw bbox
        if bbox is not None and len(bbox) >= 4:
            x, y, bw, bh = bbox[:4]
            x1, y1 = int(x), int(y)
            x2, y2 = int(x + bw), int(y + bh)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                img,
                f"ID {track_id}",
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

        # draw joints
        for x, y, c in kp:
            if c >= KP_THR:
                cv2.circle(img, (int(x), int(y)), 3, color, -1)

        # draw skeleton
        for a, b in SKELETON:
            if kp[a, 2] >= KP_THR and kp[b, 2] >= KP_THR:
                pt1 = (int(kp[a, 0]), int(kp[a, 1]))
                pt2 = (int(kp[b, 0]), int(kp[b, 1]))
                cv2.line(img, pt1, pt2, color, 2)

    writer.write(img)

writer.release()
print("Saved:", out_path)


Saved: /content/01_0028_pose_video.mp4
